<a href="https://colab.research.google.com/github/csnorbi18/Projekt/blob/master/BPI_Challenge_2017.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.0 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import pandas as pd
import pm4py
import gzip
import shutil
import os



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Feltöltött fájl:", filename)

Saving BPI Challenge 2017.xes.gz to BPI Challenge 2017.xes.gz
Feltöltött fájl: BPI Challenge 2017.xes.gz


In [10]:
if not Path(filename).is_file():
    raise FileNotFoundError(f"Nem található a fájl: {filename}")

# XES és tömörített XES fájlok kezelése
if filename.lower().endswith((".xes", ".xes.gz")):

    # A PM4Py közvetlenül támogatja a .xes.gz fájlokat
    log = pm4py.read_xes(filename)

    # Átalakítás DataFrame formátumra, ha szükséges
    if not isinstance(log, pd.DataFrame):
        log = pm4py.convert_to_dataframe(log)

    # Kötelező PM4Py-oszlopok
    required = [
        "case:concept:name",
        "concept:name",
        "time:timestamp"
    ]

    missing = [c for c in required if c not in log.columns]

    if missing:
        raise ValueError(
            "A XES fájlból hiányzó kötelező attribútum(ok): "
            + ", ".join(missing)
        )

    # Időbélyegek egységesítése
    log["time:timestamp"] = pd.to_datetime(
        log["time:timestamp"],
        utc=True
    )

    # Egységes PM4Py DataFrame-formátum
    log = pm4py.format_dataframe(
        log,
        case_id="case:concept:name",
        activity_key="concept:name",
        timestamp_key="time:timestamp"
    )

    print("XES eseménynapló sikeresen betöltve.")

else:
    raise ValueError("Csak .xes vagy .xes.gz fájl támogatott.")

# Rendezés esetszám és időbélyeg alapján
log = (
    log.sort_values(
        ["case:concept:name", "time:timestamp"],
        kind="stable"
    )
    .reset_index(drop=True)
)

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

XES eseménynapló sikeresen betöltve.
Események száma: 1,202,267
Esetek száma: 31,509
Tevékenységek száma: 26


,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,...,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,@@index,@@case_index
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,2016-08-03 15:57:21.673000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,2016-08-03 15:57:21.734000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,2016-08-03 15:57:21.963000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1030261128,withdraw,2016-08-03 15:58:28.286000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
4,Created,User_1,W_Complete application,Workflow,Workitem_1127124826,schedule,2016-08-03 15:58:28.293000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0


In [15]:
n_events = len(log)
n_cases = log["case:concept:name"].nunique()
n_activities = log["concept:name"].nunique()

print("Események száma:", n_events)
print("Esetek száma:", n_cases)
print("Tevékenységek száma:", n_activities)
print("Átlagos eseményszám esetenként:", round(n_events / n_cases, 2) if n_cases else None)

print("\nOszlopok:")
print(list(log.columns))
print("\nAdattípusok:")
print(log.dtypes)

log.head(3)

Események száma: 1202267
Esetek száma: 31509
Tevékenységek száma: 26
Átlagos eseményszám esetenként: 38.16

Oszlopok:
['Action', 'org:resource', 'concept:name', 'EventOrigin', 'EventID', 'lifecycle:transition', 'time:timestamp', 'case:LoanGoal', 'case:ApplicationType', 'case:concept:name', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount', 'OfferID', '@@index', '@@case_index']

Adattípusok:
Action                                object
org:resource                          object
concept:name                  string[python]
EventOrigin                           object
EventID                               object
lifecycle:transition                  object
time:timestamp           datetime64[ns, UTC]
case:LoanGoal                         object
case:ApplicationType                  object
case:concept:name             string[python]
case:RequestedAmount                 float64
FirstWithdrawalAmount    

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,...,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,@@index,@@case_index
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,2016-08-03 15:57:21.673000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,2016-08-03 15:57:21.734000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,2016-08-03 15:57:21.963000+00:00,"Other, see explanation",New credit,Application_1000086665,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0


In [16]:
activities = pm4py.get_event_attribute_values(log, "concept:name")
activity_table = (
    pd.Series(activities, name="gyakoriság")
    .sort_values(ascending=False)
    .to_frame()
)
activity_table

,gyakoriság
W_Validate application,209496
W_Call after offers,191092
W_Call incomplete files,168529
W_Complete application,148900
W_Handle leads,47264
O_Create Offer,42995
O_Created,42995
O_Sent (mail and online),39707
A_Validating,38816
A_Accepted,31509


In [17]:
if "org:resource" in log.columns:
    resources = pm4py.get_event_attribute_values(log, "org:resource")
    resource_table = (
        pd.Series(resources, name="gyakoriság")
        .sort_values(ascending=False)
        .to_frame()
    )
    display(resource_table)

,gyakoriság
User_1,148404
User_3,26342
User_5,22900
User_87,22498
User_30,21272
...,...
User_146,23
User_145,10
User_148,4
User_147,2


In [20]:
start_activities = pm4py.get_start_activities(log)
end_activities = pm4py.get_end_activities(log)

print("Kezdőtevékenységek:")
display(pd.Series(start_activities, name="esetek száma").sort_values(ascending=False).to_frame())

print("\nVégtevékenységek:")
display(pd.Series(end_activities, name="esetek száma").sort_values(ascending=False).to_frame())

Kezdőtevékenységek:


,esetek száma
A_Create Application,31509



Végtevékenységek:


,esetek száma
W_Validate application,12662
W_Call after offers,9457
W_Call incomplete files,4676
O_Cancelled,4436
W_Complete application,144
W_Assess potential fraud,102
O_Sent (mail and online),15
O_Sent (online only),8
W_Shortened completion,5
W_Personal Loan collection,2


In [21]:
min_time = log["time:timestamp"].min()
max_time = log["time:timestamp"].max()

print("Első időbélyeg:", min_time)
print("Utolsó időbélyeg:", max_time)
print("Teljes időtartam:", max_time - min_time)

if "start:timestamp" in log.columns:
    print("Legkorábbi kezdési idő:", log["start:timestamp"].min())

Első időbélyeg: 2016-01-01 09:51:15.304000+00:00
Utolsó időbélyeg: 2017-02-01 14:11:03.499000+00:00
Teljes időtartam: 397 days 04:19:48.195000


In [22]:
variant_table = (
    log.sort_values(["case:concept:name", "time:timestamp"])
       .groupby("case:concept:name")["concept:name"]
       .apply(tuple)
       .value_counts()
       .rename_axis("variáns")
       .reset_index(name="esetek_száma")
)

print("Folyamatvariánsok száma:", len(variant_table))
display(variant_table.head(10))

Folyamatvariánsok száma: 15930


,variáns,esetek_száma
0,"(A_Create Application, A_Submitted, W_Handle l...",1056
1,"(A_Create Application, W_Complete application,...",1021
2,"(A_Create Application, A_Submitted, W_Handle l...",734
3,"(A_Create Application, A_Submitted, W_Handle l...",451
4,"(A_Create Application, A_Submitted, W_Handle l...",332
5,"(A_Create Application, A_Submitted, W_Handle l...",298
6,"(A_Create Application, W_Complete application,...",278
7,"(A_Create Application, A_Submitted, W_Handle l...",244
8,"(A_Create Application, A_Submitted, W_Handle l...",212
9,"(A_Create Application, A_Submitted, W_Handle l...",204


In [23]:
top_n = min(10, len(variant_table))
covered = variant_table.head(top_n)["esetek_száma"].sum()
coverage = covered / n_cases if n_cases else 0

print(f"A {top_n} leggyakoribb variáns az esetek {coverage:.1%}-át fedi le.")

A 10 leggyakoribb variáns az esetek 15.3%-át fedi le.


In [ ]:
import matplotlib.pyplot as plt

plot_df = log.sort_values(
    ["case:concept:name", "time:timestamp"]
).copy()

# Az eseteket 0..N-1 értékekre képezzük
cases = plot_df["case:concept:name"].unique()
case_map = {case: i for i, case in enumerate(cases)}
plot_df["case_index"] = plot_df["case:concept:name"].map(case_map)

activities = sorted(plot_df["concept:name"].dropna().unique())

fig, ax = plt.subplots(figsize=(14, 8))

for activity in activities:
    part = plot_df[plot_df["concept:name"] == activity]

    ax.scatter(
        part["time:timestamp"],
        part["case_index"],
        s=12,
        label=activity
    )

ax.set_title("Eseménynapló – dotted chart")
ax.set_xlabel("Idő")
ax.set_ylabel("Eset")

ax.legend(
    title="Tevékenység",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    ncol=2,
    fontsize=8
)

plt.tight_layout()
plt.show()